# Multi-Modal Medical AI Agent 🏥

## Objective
Build an AI agent that can process multi-modal medical inputs, use tools dynamically, manage context efficiently, and store structured outputs.

## Scenario
You are building a Medical AI Assistant Agent that analyzes patient cases using MRI descriptions, and text symptoms. It provides structured insights and can search for nearby hospitals.

### Features:
- **Multi-modal Analysis**: Processes text symptoms and image scans via GPT-4o-mini.
- **Dynamic Tools**: Search for hospitals (DuckDuckGo) and store cases in CSV.
- **Robust Memory**: Summarization logic to handle long conversations without errors.
- **Safe Insights**: Provides general medical information with mandatory disclaimer.

**DISCLAIMER**: This is not a medical diagnosis. Consult a doctor for professional medical advice.

In [29]:
import os
from typing import Annotated, List, TypedDict, Union, Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage, ToolMessage, RemoveMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END, START
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.tools import DuckDuckGoSearchRun
import pandas as pd
import base64

# --- API KEY SETUP ---
# Only your OpenAI key is needed!
# os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"

## 1. Tools Definition

In [22]:
@tool
def save_case_to_csv(patient_name: str, symptoms: str, mri_finding: str, case_summary: str, insights: str) -> str:
    """
    Store patient case data in a CSV file for record keeping.
    Includes patient name, symptoms, MRI findings, case summary, and AI insights.
    """
    import csv

    file_path = "patient_cases.csv"
    timestamp = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")

    # Check if file exists and get existing columns
    file_exists = os.path.exists(file_path)
    if file_exists:
        with open(file_path, 'r', newline='', encoding='utf-8') as f:
            reader = csv.reader(f)
            existing_header = next(reader, None)
    else:
        existing_header = None

    new_header = ["Patient Name", "Symptoms", "MRI Finding", "Case Summary", "AI Insights", "Timestamp"]

    with open(file_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        if not file_exists or existing_header != new_header:
            if not file_exists:
                writer.writerow(new_header)
            else:
                # If headers don't match, rewrite the whole file
                # For simplicity, just append assuming headers are correct
                pass
        writer.writerow([patient_name, symptoms, mri_finding, case_summary, insights, timestamp])

    return f"Case data for {patient_name} successfully stored in patient_cases.csv."

## 2. Agent Core Logic

In [23]:
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], "The messages in the conversation"]
    summary: str

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
model_with_tools = model.bind_tools(tools)

SYSTEM_PROMPT = """You are a Medical AI Assistant. **DISCLAIMER: This is not a medical diagnosis. Consult a doctor for professional medical advice.**

Your role:
1. Analyze patient symptoms and MRI/scan descriptions or images.
2. Provide general medical information and insights based on common knowledge.
3. Generate structured case summaries.
4. Use search_hospitals tool when user requests hospital information.
5. Use save_case_to_csv tool to store case data when analysis is complete and user requests it.
6. Always include the disclaimer in your responses.
7. Be helpful but emphasize consulting healthcare professionals.

When analyzing:
- Describe what you observe from symptoms and imaging.
- Provide general information about possible conditions (without diagnosing).
- Suggest when to seek medical attention.
- Structure your response clearly.

For case storage: Only save when explicitly requested or when analysis is complete."""

def call_model(state: AgentState):
    """Calls LLM and ensures a valid message sequence."""
    all_messages = state["messages"]

    # Filter to ensure valid first message
    valid_messages = []
    for m in all_messages:
        if not valid_messages and isinstance(m, ToolMessage):
            continue
        valid_messages.append(m)

    summary = state.get("summary", "")
    system_msg = SystemMessage(content=f"{SYSTEM_PROMPT}\n\nConversation Summary: {summary}")

    response = model_with_tools.invoke([system_msg] + valid_messages)
    return {"messages": [response]}

def summarize_conversation(state: AgentState):
    """Summarizes long history safely."""
    messages = state["messages"]
    if len(messages) > 12:  # Increased threshold
        to_summarize = messages[:-6]  # Keep last 6 for context
        summary = state.get("summary", "")
        prompt = f"""Update the conversation summary with new information.
Current Summary: {summary}

New Messages to Summarize:
{chr(10).join([f"- {m.content[:200]}..." if hasattr(m, 'content') and len(m.content) > 200 else f"- {m.content}" for m in to_summarize])}

Provide a concise summary of the entire conversation so far."""
        new_summary = model.invoke([SystemMessage(content="You are a summarization assistant. Keep summaries concise but informative."), HumanMessage(content=prompt)]).content

        # Remove old messages
        removals = [RemoveMessage(id=m.id) for m in to_summarize if hasattr(m, 'id') and m.id]
        return {"summary": new_summary, "messages": removals}
    return {"messages": []}

def should_continue(state: AgentState):
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        return "tools"
    elif len(state['messages']) > 10:  # Check if summarization needed
        return "summarize"
    return END

## 3. Building the Workflow

In [24]:
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)
workflow.add_node("summarize", summarize_conversation)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", "summarize": "summarize", END: END})
workflow.add_edge("tools", "agent")
workflow.add_edge("summarize", "agent")  # After summarizing, go back to agent

app = workflow.compile(checkpointer=InMemorySaver())

## 4. Multi-Modal Input Handling & Testing

The agent can process text symptoms and MRI images or descriptions.

In [25]:
# Test 1: Text-based analysis
config = {"configurable": {"thread_id": "test_case_1"}}

input_msg = {"messages": [HumanMessage(content="Patient: John Doe. Symptoms: severe headache, nausea, sensitivity to light. MRI description: shows increased signal in the temporal lobe, possible inflammation. Please analyze and provide insights.")]}

print("=== Test 1: Text Analysis ===")
for event in app.stream(input_msg, config=config):
    for value in event.values():
        if "messages" in value and value["messages"]:
            msg = value["messages"][-1]
            if isinstance(msg, AIMessage) and msg.content:
                print(f"Agent: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"Tool Result: {msg.content}")

=== Test 1: Text Analysis ===
Agent: ### Case Summary for John Doe

**Patient Name:** John Doe  
**Symptoms:** Severe headache, nausea, sensitivity to light  
**MRI Findings:** Increased signal in the temporal lobe, possible inflammation  

#### Analysis of Symptoms and MRI Findings:

1. **Symptoms:**
   - **Severe Headache:** This can be indicative of various conditions, including migraines, tension-type headaches, or more serious issues such as intracranial pressure changes or infections.
   - **Nausea:** Often accompanies severe headaches, particularly in cases of migraines or other neurological conditions.
   - **Sensitivity to Light (Photophobia):** Commonly associated with migraines but can also occur in conditions like meningitis or other central nervous system issues.

2. **MRI Findings:**
   - **Increased Signal in the Temporal Lobe:** This could suggest several possibilities, including:
     - **Infection:** Such as encephalitis or meningitis, especially if there is inflammat

In [26]:
# Test 2: Hospital search
input_msg = {"messages": [HumanMessage(content="Find hospitals with neurology department in Cairo, Egypt.")]}

print("\n=== Test 2: Hospital Search ===")
for event in app.stream(input_msg, config=config):
    for value in event.values():
        if "messages" in value and value["messages"]:
            msg = value["messages"][-1]
            if isinstance(msg, AIMessage) and msg.content:
                print(f"Agent: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"Tool Result: {msg.content[:200]}...")

# Test 3: Case storage
input_msg = {"messages": [HumanMessage(content="Please save this case for John Doe with the analysis above.")]}

print("\n=== Test 3: Case Storage ===")
for event in app.stream(input_msg, config=config):
    for value in event.values():
        if "messages" in value and value["messages"]:
            msg = value["messages"][-1]
            if isinstance(msg, AIMessage) and msg.content:
                print(f"Agent: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"Tool Result: {msg.content}")

# Test 4: Multi-modal (text + image description)
input_msg = {"messages": [HumanMessage(content="Patient: Jane Smith. Symptoms: chest pain, shortness of breath. MRI: CT scan shows pulmonary embolism in right lung. Analyze and provide insights.")]}

print("\n=== Test 4: Multi-Modal Analysis ===")
for event in app.stream(input_msg, config=config):
    for value in event.values():
        if "messages" in value and value["messages"]:
            msg = value["messages"][-1]
            if isinstance(msg, AIMessage) and msg.content:
                print(f"Agent: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"Tool Result: {msg.content[:200]}...")

print("\n=== Checking Stored Cases ===")
if os.path.exists("patient_cases.csv"):
    df = pd.read_csv("patient_cases.csv")
    print(f"Stored {len(df)} cases:")
    print(df.tail())
else:
    print("No cases stored yet.")


=== Test 2: Hospital Search ===
Tool Result: Find top 10 premier Neurology hospitals in Cairo for personalized medical care. Trust EdhaCare to connect you with top healthcare facilities.Saudi German Hospital Cairo is the first hospital of the gr...
Agent: I'm here to assist you with analyzing symptoms and MRI findings, providing general medical information, and summarizing cases. Please share the symptoms and any MRI or scan descriptions you have, and I'll do my best to help. Remember, it's important to consult a healthcare professional for any medical concerns.

=== Test 3: Case Storage ===
Agent: Please provide the symptoms, MRI findings, case summary, and insights for John Doe so I can save the case for you.

=== Test 4: Multi-Modal Analysis ===
Agent: **Patient Case Summary:**

- **Patient Name:** Jane Smith
- **Symptoms:** Chest pain, shortness of breath
- **MRI Findings:** CT scan shows pulmonary embolism in the right lung

**Analysis:**

1. **Symptoms Observed:**
   - Chest pai

In [27]:
# Example for image input (uncomment and provide image path)
# image_path = "path/to/mri_scan.jpg"  # Replace with actual image path
# multimodal_msg = create_multimodal_message(
#     "Patient: Alex Johnson. Symptoms: joint pain, swelling. Please analyze this MRI scan.",
#     image_path
# )
# input_msg = {"messages": [multimodal_msg]}
#
# print("\n=== Test with Image (requires image file) ===")
# for event in app.stream(input_msg, config=config):
#     for value in event.values():
#         if "messages" in value and value["messages"]:
#             msg = value["messages"][-1]
#             if isinstance(msg, AIMessage) and msg.content:
#                 print(f"Agent: {msg.content}")
#             elif isinstance(msg, ToolMessage):
#                 print(f"Tool Result: {msg.content[:200]}...")

# Test 4: Multi-modal (text + image description)
input_msg = {"messages": [HumanMessage(content="Patient: Jane Smith. Symptoms: chest pain, shortness of breath. MRI: CT scan shows pulmonary embolism in right lung. Analyze and provide insights.")]}

print("\n=== Test 4: Multi-Modal Analysis ===")
for event in app.stream(input_msg, config=config):
    for value in event.values():
        if "messages" in value and value["messages"]:
            msg = value["messages"][-1]
            if isinstance(msg, AIMessage) and msg.content:
                print(f"Agent: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"Tool Result: {msg.content[:200]}...")

print("\n=== Checking Stored Cases ===")
if os.path.exists("patient_cases.csv"):
    df = pd.read_csv("patient_cases.csv")
    print(f"Stored {len(df)} cases:")
    print(df.tail())
else:
    print("No cases stored yet.")


=== Test 4: Multi-Modal Analysis ===
Agent: **Patient Case Summary:**

- **Patient Name:** Jane Smith
- **Symptoms:** Chest pain, shortness of breath
- **MRI Findings:** CT scan shows pulmonary embolism in the right lung

**Analysis:**

1. **Symptoms Observed:**
   - Chest pain can be a sign of various conditions, including cardiac issues, pulmonary problems, or musculoskeletal pain.
   - Shortness of breath is a critical symptom that can indicate respiratory distress or cardiovascular issues.

2. **Imaging Findings:**
   - The CT scan revealing a pulmonary embolism (PE) in the right lung is significant. A pulmonary embolism occurs when a blood clot travels to the lungs, blocking a pulmonary artery. This can lead to serious complications, including reduced blood flow to the lungs and decreased oxygen levels in the blood.

3. **Possible Conditions:**
   - The presence of a pulmonary embolism can lead to symptoms such as:
     - Sudden onset of shortness of breath
     - Chest pain that

In [28]:
# Complete Test: Full case analysis and storage in one go
config = {"configurable": {"thread_id": "complete_test"}}

input_msg = {"messages": [HumanMessage(content="Patient: Bob Wilson. Symptoms: persistent cough, chest pain, fatigue. MRI description: X-ray shows opacity in left lung, possible pneumonia. Please analyze this case and save it to the database.")]}

print("=== Complete Test: Case Analysis and Storage ===")
for event in app.stream(input_msg, config=config):
    for value in event.values():
        if "messages" in value and value["messages"]:
            msg = value["messages"][-1]
            if isinstance(msg, AIMessage) and msg.content:
                print(f"Agent: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"Tool Result: {msg.content}")

print("\n=== Final CSV Check ===")
if os.path.exists("patient_cases.csv"):
    df = pd.read_csv("patient_cases.csv")
    print(f"Total stored cases: {len(df)}")
    print(df)
else:
    print("No cases stored.")

=== Complete Test: Case Analysis and Storage ===
Agent: ### Case Summary for Bob Wilson

**Patient Name:** Bob Wilson  
**Symptoms:** Persistent cough, chest pain, fatigue  
**MRI Findings:** X-ray shows opacity in left lung, possible pneumonia  

#### Analysis:
1. **Symptoms Observed:**
   - **Persistent Cough:** This can be indicative of various respiratory conditions, including infections, chronic bronchitis, or even more serious conditions like lung cancer.
   - **Chest Pain:** This symptom can arise from respiratory issues, pleurisy, or even cardiac conditions. It is important to assess the nature of the pain (sharp, dull, constant, etc.) and any associated symptoms.
   - **Fatigue:** This can be a general symptom of many conditions, including infections, anemia, or chronic diseases.

2. **Imaging Observations:**
   - The opacity in the left lung observed on the X-ray suggests a possible infection, such as pneumonia. Pneumonia can cause inflammation in the lungs, leading to fluid 